In [31]:
import pandas as pd
import re
import csv
import pytesseract
from tempfile import TemporaryDirectory
from PIL import Image
from pathlib import Path 
from pdf2image import convert_from_path


home = Path()/".."
pdf = home/"data/bini_dict.pdf"


In [33]:
def read_pdf2img(pdf_path):
    """
    Converts the PDF to images and writes the OCR output to a single text file.
    """
    output_folder = home/"data"
    output_folder.mkdir(exist_ok=True)

    with TemporaryDirectory() as tmp:
        storage = Path(tmp)
        convert_from_path(
            pdf_path=Path(pdf_path),
            output_folder=storage,
            fmt='png',
            single_file=False,
            first_page=19, 
        )

        with open(output_folder/"bini_output.txt", "w") as f:
            for x in sorted(storage.glob("*.png")): 
                text = pytesseract.image_to_string(Image.open(x), lang='bini') # after training the model, use lang=['eng', 'bini'] as an argument
                f.write(text + "\n")  # Add newlines between pages
                # fix: pages of the file ebing written get scattered, find out why


read_pdf2img(pdf)

In [ ]:
images = Path("../pdf_img")
output_dir = Path("../output_folder")

for x in images.iterdir():
    name = x.stem
    img = Image.open(x)
    content = pytesseract.image_to_string(img, lang="bini")
    with open(f"{output_dir}/{name}.txt", "w") as f:
        f.write(content)


In [34]:
input_txt   = "../data/output_file.txt"
bini_txt    = "../data/bini_output.txt"
bini_output = "../data/proper_bini_ouput.txt"
bini_csv    = "../data/proper_bini_output.csv"
output_txt  = "../data/bini_words_definitions.txt"
output_csv  = "../data/bini_words_definitions.csv"

with open(bini_txt, "r", encoding="utf-8") as f:
    text = f.read()

# removing unneeded things from the corpus
# text = re.sub(r"\BINI DICTIONARY", "", text)  # remove BINI DICTIONARY
# text = re.sub(r"\2|\4|\6|\7|\9", "", text) # remove certain numbers
# text = re.sub(r"\»", "", text)
# text = re.sub(r"\™", "", text)

# # ---substituting certain values for mistaken counterparts
# text = re.sub(r"\3", "ε", text)
# text = re.sub(r"\1", "i", text)
# # text = re.sub(r"\5", "", text)
# text = re.sub(r"\8", "", text)
# text = re.sub(r"\¢", "c", text)
# text = re.sub(r"\¥", "Y", text)
# # text = re.sub(r"\#", "", text)
# text = re.sub(r"\€", "ε", text)
# text = re.sub(r"\©", "c", text)

# Pattern: word at start of line, then phonetic, then definition
pattern = re.compile(
    r"(?m)^([a-zA-Zɣεↄɽῦυ]+)"           # word
    r"\s+\[.*?\]"                 # phonetic (skip)
    r"\s+(.*?)(?=(?:\n[A-Za-zɣεↄɽῦυ]+ \[)|\Z)",  # definition until next entry or end
    re.S
)

with open(bini_output, "w", encoding="utf-8") as out:
    for match in pattern.finditer(text):
        word = match.group(1).strip()
        definition = match.group(2).replace('\n', ' ').strip()
        out.write(f"{word}: {definition}\n")

with open(bini_output, "r", encoding="utf-8") as f:
    lines = f.readlines()

with open(bini_csv, "w", encoding="utf-8", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["word", "definition"])
    for line in lines:
        if ": " in line:
            word, definition = line.strip().split(": ", 1)
            writer.writerow([word, definition])


In [35]:
df = pd.read_csv(bini_csv)

df.sample(30)

,word,definition
2800,vi,(q to open (of a sore only). (2) to come out (...
2612,wahε,"and Igbaɣ> [...] when they are ""travelling"" ; ..."
502,ekuzo,"a shrub, Omgokea Rlaineana;, cf. eka [.](""); u..."
927,gu,gua [.. to talk with somebody. gu [.] gwi [.]...
466,ebhaya,"hire; rent; ehaya umu_ikεks ue yi ca [...... ""..."
2378,oxuo,"to marry a woman; dɽↄ€ῦ-ↄε [...""] (a) she marr..."
813,gveεe,kola; Cola acuminata-- verticilata; Tv-oha [.....
1588,gbu,"""to sweep the sweep- ings"": to do the last par..."
2131,ikpgsi,"only, i.e. without any £ɣε [..] (and without n..."
2339,rhurhurhu,staggering; tumb- ling against things; ɽu rhurhu.


In [36]:
print(df.count())

word          2978
definition    2978
dtype: int64


In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2978 entries, 0 to 2977
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   word        2978 non-null   object
 1   definition  2978 non-null   object
dtypes: object(2)
memory usage: 46.7+ KB
